In [ ]:
# %load_ext watermark
# %watermark -a "Paul A. Wambo" -v -p Bio,matplotlib,numpy,pandas,pyMIBiG,seaborn,sklearn,transformers,torch

In [ ]:
import os

import torch

import numpy as np
import pandas as pd
import pickle as pkl
import seaborn as sns
import matplotlib.pyplot as plt

from concurrent.futures import ThreadPoolExecutor

# from Bio import SeqIO
from transformers import AutoTokenizer, AutoModel, AutoModelForMaskedLM

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
datapath = "/content/drive/MyDrive/criblage_biosciences/data/output"
with open(os.path.join(datapath, "parsed_mibig_gene.pkl"), "rb") as f:
  df = pkl.load(f)
df.head()

,BGC_ID,Protein_ID,Protein_Name,DNA_Seq,DNA_Embedding,Protein_Seq,Protein_Embedding
0,BGC0000001,AEK75490.1,protein methyltransferase,ATGACTGAATTAGATCGTGCTTTTGATGCTGTTCCTGCTCCTATTT...,[-0.09805445373058319],MTELDRAFDAVPAPIYTHHERHGETVHRSAPESIRRELAALQVRAG...,"[[0.2848623991012573, 0.05677460879087448, -0...."
1,BGC0000001,AEK75491.1,putative hydrolase,ATGAAACCTCCTGCTTCTTCTGTTTGTCCTGTTGATACTTCTAAAA...,[-0.1703575998544693],MKPPASSVCPVDTSKMGNRSSPARYGRRPRKRCVELSETNLEFVHV...,"[[0.15280525386333466, -0.003178832121193409, ..."
2,BGC0000001,AEK75492.1,pathway-specific SARP activator,ATGTTAGGTGCTTTACGTATTGCTGATCCTGCTCCTCGTACTATTA...,[-0.12983764708042145],MLGALRIADPAPRTITAPKVETLFATLLIRANHTVTTDELIAELWG...,"[[0.1949358582496643, 0.03870860114693642, 0.0..."
3,BGC0000001,AEK75493.1,cytochrome P450,ATGACTGATGTTCAATTACCTGCTTTTCCTATGACTCGTACTTGTC...,[-0.162684828042984],MTDVQLPAFPMTRTCPHQPPEGYAALRENGPLAQVRLVGDRTAWVV...,"[[0.15936234593391418, 0.0026949350722134113, ..."
4,BGC0000001,AEK75494.1,LuxR family transcriptional regulator,ATGCGTGATACTGCTGAACATCGTATTGGTACTTCTGGTCGTCATA...,[-0.10447891801595688],MRDTAEHRIGTSGRHTPQAQATPADRLSQALARARSGRGGVVELVG...,"[[0.2321871817111969, -0.031208429485559464, -..."


In [ ]:
saved_paths = [os.path.join(datapath, x) for x in os.listdir(datapath) if '35M' in x and x.endswith('.pkl')]

_df = []

for _, path in enumerate(saved_paths):
    with open(path, "rb") as f:
        _datum = pkl.load(f)
        _df.append(_datum)

df_embeddings = pd.concat(_df, ignore_index=True)
df_embeddings.head()

,bgc_id,protein_name,protein_id,protein_sequence,dna_sequence,embeddings,sequence,Biosynthetic Class,bgc_sequence
0,BGC0000001,orfP,AEK75490.1,MTELDRAFDAVPAPIYTHHERHGETVHRSAPESIRRELAALQVRAG...,GTGACCGAGCTCGACCGGGCCTTCGACGCCGTACCGGCCCCGATCT...,"[[tensor(0.2849), tensor(0.0568), tensor(-0.03...",NaN,NaN,NaN
1,BGC0000001,None,AEK75491.1,MKPPASSVCPVDTSKMGNRSSPARYGRRPRKRCVELSETNLEFVHV...,GTGAAGCCGCCGGCCAGCTCTGTCTGCCCGGTGGACACCTCGAAGA...,"[[tensor(0.1528), tensor(-0.0032), tensor(0.20...",NaN,NaN,NaN
2,BGC0000001,abyR,AEK75492.1,MLGALRIADPAPRTITAPKVETLFATLLIRANHTVTTDELIAELWG...,TTGCTCGGAGCATTGCGGATCGCCGATCCCGCACCTCGCACGATTA...,"[[tensor(0.1949), tensor(0.0387), tensor(0.077...",NaN,NaN,NaN
3,BGC0000001,abyX,AEK75493.1,MTDVQLPAFPMTRTCPHQPPEGYAALRENGPLAQVRLVGDRTAWVV...,ATGACCGACGTCCAGCTGCCCGCGTTCCCGATGACCCGCACCTGTC...,"[[tensor(0.1594), tensor(0.0027), tensor(-0.01...",NaN,NaN,NaN
4,BGC0000001,abyH,AEK75494.1,MRDTAEHRIGTSGRHTPQAQATPADRLSQALARARSGRGGVVELVG...,GTGCGCGACACGGCAGAGCATCGGATCGGCACATCCGGGCGTCACA...,"[[tensor(0.2322), tensor(-0.0312), tensor(-0.0...",NaN,NaN,NaN


In [ ]:
df_embeddings = pd.merge(df_embeddings, df[['Protein_ID', 'Protein_Name']], left_on='protein_id', right_on='Protein_ID', how='left')
df_embeddings.drop(columns=['Protein_ID'], inplace=True)
df_embeddings.head()

,bgc_id,protein_name,protein_id,protein_sequence,dna_sequence,embeddings,sequence,Biosynthetic Class,bgc_sequence,Protein_Name
0,BGC0000001,orfP,AEK75490.1,MTELDRAFDAVPAPIYTHHERHGETVHRSAPESIRRELAALQVRAG...,GTGACCGAGCTCGACCGGGCCTTCGACGCCGTACCGGCCCCGATCT...,"[[tensor(0.2849), tensor(0.0568), tensor(-0.03...",NaN,NaN,NaN,protein methyltransferase
1,BGC0000001,None,AEK75491.1,MKPPASSVCPVDTSKMGNRSSPARYGRRPRKRCVELSETNLEFVHV...,GTGAAGCCGCCGGCCAGCTCTGTCTGCCCGGTGGACACCTCGAAGA...,"[[tensor(0.1528), tensor(-0.0032), tensor(0.20...",NaN,NaN,NaN,putative hydrolase
2,BGC0000001,abyR,AEK75492.1,MLGALRIADPAPRTITAPKVETLFATLLIRANHTVTTDELIAELWG...,TTGCTCGGAGCATTGCGGATCGCCGATCCCGCACCTCGCACGATTA...,"[[tensor(0.1949), tensor(0.0387), tensor(0.077...",NaN,NaN,NaN,pathway-specific SARP activator
3,BGC0000001,abyX,AEK75493.1,MTDVQLPAFPMTRTCPHQPPEGYAALRENGPLAQVRLVGDRTAWVV...,ATGACCGACGTCCAGCTGCCCGCGTTCCCGATGACCCGCACCTGTC...,"[[tensor(0.1594), tensor(0.0027), tensor(-0.01...",NaN,NaN,NaN,cytochrome P450
4,BGC0000001,abyH,AEK75494.1,MRDTAEHRIGTSGRHTPQAQATPADRLSQALARARSGRGGVVELVG...,GTGCGCGACACGGCAGAGCATCGGATCGGCACATCCGGGCGTCACA...,"[[tensor(0.2322), tensor(-0.0312), tensor(-0.0...",NaN,NaN,NaN,LuxR family transcriptional regulator


In [ ]:
df_embeddings.shape

(4001629, 10)

In [ ]:
df_embeddings['Protein_Name'].fillna('Annotation Not Available', inplace=True)

/tmp/ipython-input-1656313781.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_embeddings['Protein_Name'].fillna('Annotation Not Available', inplace=True)


In [ ]:
df_embeddings['Protein_Name'].value_counts().to_csv(os.path.join(datapath, 'protein_counts.csv'))

In [ ]:
import re

# ============================================================
# 1. Natural Product Biosynthetic Family Classification
# ============================================================

def classify_np_family(name):
    n = name.lower()

    # Core biosynthetic systems
    if "nrps" in n or "nonribosomal" in n or "adenylation" in n or "a-domain" in n:
        return "NRPS"
    if "pks" in n or "polyketide" in n or "ketosynthase" in n or "ks-domain" in n:
        return "PKS"
    if "terpene" in n or "terpenoid" in n or "isoprenyl" in n:
        return "Terpene Synthase"
    if "fatty acid" in n or "fab" in n or "fas" in n:
        return "Fatty Acid Synthase (FAS)"

    # Hybrid systems
    if "nrps" in n and "pks" in n:
        return "Hybrid PKS-NRPS"

    # Tailoring enzymes
    if any(k in n for k in [
        "methyltransferase", "oxidase", "reductase", "hydroxylase", "glycosyltransferase",
        "dehydrogenase", "oxygenase", "cyclase", "halogenase"
    ]):
        return "Tailoring Enzyme"

    # Transporters
    if any(k in n for k in ["transporter", "abc transporter", "efflux", "membrane protein"]):
        return "Transporter"

    # Regulators
    if any(k in n for k in ["regulator", "transcription factor", "sigma factor", "two-component"]):
        return "Regulator"

    # Hypothetical
    if "hypothetical" in n or "unknown" in n:
        return "Hypothetical/Unknown"

    # Default
    return "Other"


# ============================================================
# 2. EC Class Assignment (1–6)
# ============================================================

EC_CLASSES = {
    "Oxido-reductase": ["oxidoreductase", "dehydrogenase", "oxidase", "reductase"],
    "Transferase": ["transferase", "kinase", "methyltransferase", "acetyltransferase", "glycosyltransferase"],
    "Hydrolase": ["hydrolase", "protease", "peptidase", "esterase", "lipase", "nuclease"],
    "Lyase": ["lyase", "synthase", "decarboxylase", "aldolase"],
    "Isomerase": ["isomerase", "epimerase", "racemase"],
    "Synthetase": ["ligase", "synthetase"],
}

def classify_ec_class(name):
    n = name.lower()

    # direct EC number present?
    ec_match = re.search(r"ec[:\s]*(\d)\.", n)
    if ec_match:
        return f"{ec_match.group(1)}"

    # keyword mapping
    for ec, keywords in EC_CLASSES.items():
        if any(k in n for k in keywords):
            return f"EC {ec}"

    return "EC Unknown"

In [ ]:
df_embeddings['np_labels'] = df_embeddings['Protein_Name'].apply(classify_np_family)
df_embeddings['np_labels'].value_counts()

,count
np_labels,
Other,3367551
Hypothetical/Unknown,324661
Tailoring Enzyme,91414
Transporter,76404
PKS,68288
Regulator,46631
NRPS,26510
Fatty Acid Synthase (FAS),89
Terpene Synthase,81


In [ ]:
df_embeddings['ec_labels'] = df_embeddings['Protein_Name'].apply(classify_ec_class)
df_embeddings['ec_labels'].value_counts()

,count
ec_labels,
EC Unknown,3625237
EC Lyase,99532
EC Transferase,85309
EC Oxido-reductase,69742
EC Hydrolase,53330
EC Synthetase,53206
EC Isomerase,15267
6,3
1,2


In [ ]:
df_embeddings[['protein_id', 'Protein_Name', 'np_labels', 'ec_labels']].to_parquet(os.path.join(datapath, 'mibig_proteins_labels_v2.parquet'))


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
from transformers import T5Tokenizer, T5EncoderModel

download_dir = "C:/Users/Yvael/Desktop/polo/data/rothlab"

tokenizer = T5Tokenizer.from_pretrained(
    "Rostlab/prot_t5_xl_uniref50",
    do_lower_case=False,
    cache_dir=download_dir
)

model = T5EncoderModel.from_pretrained(
    "Rostlab/prot_t5_xl_uniref50",
    cache_dir=download_dir
)


In [ ]:
model.to(device)

In [ ]:
from tqdm import tqdm

MAX_LEN_PROT = 1024
STRIDE = 512


def chunk_sequence(seq, chunk_size=MAX_LEN_PROT, stride=STRIDE):
    """Yield overlapping chunks of a protein sequence."""
    for start in range(0, len(seq), chunk_size - stride):
        yield seq[start:start + chunk_size]


def get_protein_embedding(seq, model, tokenizer, device="cuda", batch_size=8):
    """
    Compute embedding for a single protein sequence using encoder-only ProtT5.
    Optimized for GPU throughput via chunk batching.
    """
    # Insert spaces between amino acids (ProtT5 requirement)
    seq = " ".join(seq)

    # Generate chunks
    chunks = list(chunk_sequence(seq))
    n_chunks = len(chunks)

    # Precompute chunk lengths (GPU tensor)
    chunk_lengths = torch.tensor(
        [len(c) for c in chunks],
        dtype=torch.float32,
        device=device
    )

    # Store chunk embeddings here
    chunk_vecs = []

    # Process chunks in batches to maximize GPU throughput
    for i in range(0, n_chunks, batch_size):
        batch = chunks[i:i + batch_size]

        # Tokenize batch on CPU → move to GPU once
        tokens = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LEN_PROT
        )
        tokens = {k: v.to(device, non_blocking=True) for k, v in tokens.items()}

        with torch.inference_mode():
            out = model.encoder(**tokens)
            # Mean pool each sequence (batch_size × hidden_dim)
            batch_vecs = out.last_hidden_state.mean(dim=1)

        chunk_vecs.append(batch_vecs)

    # Concatenate all chunk vectors on GPU
    chunk_vecs = torch.cat(chunk_vecs, dim=0)  # shape: (num_chunks, hidden_dim)

    # Weighted average pooling across chunks
    protein_embedding = (chunk_vecs * chunk_lengths[:, None]).sum(0) / chunk_lengths.sum()

    return protein_embedding  # remains on GPU


In [ ]:
df = pd.read_csv("C:/Users/Yvael/Desktop/polo/data/input/processed_proteins_mibig.csv")
df.head()

In [ ]:
def compute_bgc_embeddings(sequences, model, tokenizer, device="cuda", batch_size=8,
                           return_numpy=False):
    seq_lengths = torch.tensor([len(s) for s in sequences],
                               dtype=torch.float32, device=device)
    protein_embeddings = [
        get_protein_embedding(seq, model, tokenizer, device=device, batch_size=batch_size)
        for seq in sequences
    ]

    stacked = torch.stack(protein_embeddings)  # GPU
    bgc_embedding = (stacked * seq_lengths[:, None]).sum(0) / seq_lengths.sum()

    if return_numpy:
        protein_embeddings_np = [emb.cpu().numpy() for emb in protein_embeddings]
        bgc_embedding_np = bgc_embedding.cpu().numpy()
        return bgc_embedding_np, protein_embeddings_np

    # Otherwise return GPU tensors
    return bgc_embedding, protein_embeddings

In [ ]:
from torch.utils.data import DataLoader

sequences = df['Protein_Sequences'].tolist()

all_bgc_embeddings, all_protein_embeddings = [], []

for seqs in tqdm(sequences, total=len(sequences)):
    bgc_embs, protein_embs = compute_bgc_embeddings(seqs, model, tokenizer, device=device)
    all_bgc_embeddings.append(bgc_embs)
    all_protein_embeddings.append(protein_embs)


In [ ]:
df['bgc_embedding'] = all_bgc_embeddings
df['protein_embedding'] = all_protein_embeddings

In [ ]:
df.head()

In [ ]:
import pickle as pkl